# Notebook 1: Downloading GSE181919 Dataset

This notebook helps you obtain the data for reproducing:

> Jarwal et al. (2024) *A deep learning method for classification of HNSCC and HPV patients using single-cell transcriptomics.* Frontiers in Molecular Biosciences 11:1395721.

## Required Files

Download these two supplementary files from the GEO page:  
**https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE181919**

Scroll to the bottom of the page under **"Supplementary file"** and download:

| File | Description | Approx. Size |
|------|-------------|-------------|
| `GSE181919_UMI_counts.txt.gz` | Gene × cell UMI count matrix (20,000 genes × 54,239 cells) | ~150 MB |
| `GSE181919_Barcode_metadata.txt.gz` | Per-cell metadata (patient, tissue type, HPV status, cell type) | ~1 MB |

Place both files in a `data/` folder next to this notebook.

In [1]:
import os

DATA_DIR = "/home/anupam/data/scRNAseq"
os.makedirs(DATA_DIR, exist_ok=True)

expected_files = [
    "GSE181919_UMI_counts.txt.gz",
    "GSE181919_Barcode_metadata.txt.gz",
]

print("Checking for required files...\n")
all_found = True
for f in expected_files:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"  OK: {f}  ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {f}")
        all_found = False

if all_found:
    print("\nAll files present. Proceed to Notebook 2.")
else:
    print(f"\nDownload missing files from GEO and place in '{os.path.abspath(DATA_DIR)}/'.")

Checking for required files...

  OK: GSE181919_UMI_counts.txt.gz  (122.0 MB)
  OK: GSE181919_Barcode_metadata.txt.gz  (0.3 MB)

All files present. Proceed to Notebook 2.


## Quick Data Inspection

In [2]:
import pandas as pd

meta_path = os.path.join(DATA_DIR, "GSE181919_Barcode_metadata.txt.gz")
meta = pd.read_csv(meta_path, sep='\t', index_col=0)

print(f"Total cells: {len(meta)}")
print(f"Columns: {meta.columns.tolist()}\n")

print("=== Cells per tissue type ===")
print(meta['tissue.type'].value_counts())

print("\n=== Tissue type × HPV cross-tab ===")
print(pd.crosstab(meta['tissue.type'], meta['hpv']))

print("\n=== Unique samples per tissue type ===")
for tt in sorted(meta['tissue.type'].unique()):
    samples = meta[meta['tissue.type']==tt]['sample.id'].nunique()
    print(f"  {tt}: {samples} samples")

meta.head()

Total cells: 54239
Columns: ['patient.id', 'sample.id', 'Gender', 'Age', 'tissue.type', 'subsite', 'hpv', 'cell.type']

=== Cells per tissue type ===
tissue.type
CA    23088
NL    16420
LN     8204
LP     6527
Name: count, dtype: int64

=== Tissue type × HPV cross-tab ===
hpv          HPV+   HPV-
tissue.type             
CA           9586  13502
LN           6548   1656
LP              0   6527
NL           5813  10607

=== Unique samples per tissue type ===
  CA: 20 samples
  LN: 4 samples
  LP: 4 samples
  NL: 9 samples


,patient.id,sample.id,Gender,Age,tissue.type,subsite,hpv,cell.type
AAACGGGCATGACGGA-1,P4,C04,F,58,CA,OC,HPV-,T.cells
AAAGATGAGCAGACTG-1,P4,C04,F,58,CA,OC,HPV-,T.cells
AAAGATGAGTGTACTC-1,P4,C04,F,58,CA,OC,HPV-,Malignant.cells
AAAGATGCACTCTGTC-1,P4,C04,F,58,CA,OC,HPV-,Malignant.cells
AAAGCAAAGACAGGCT-1,P4,C04,F,58,CA,OC,HPV-,Malignant.cells


## Dataset Summary

From the paper, the 37 tissue specimens break down as:

| Tissue Type | Code | Samples | Used in Study |
|-------------|------|---------|---------------|
| Normal | NL | 9 | Yes |
| Primary HNSCC | CA | 20 (13 HPV-, 7 HPV+) | Yes |
| Leukoplakia | LP | 4 | No |
| Metastasised | LN | 4 | No |

Notebook 2 filters to **NL + CA** only (29 samples).

## Next Steps

Proceed to **`02_hnscc_classification.ipynb`** for the full pipeline.